## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
from math import ceil

%matplotlib inline

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [2]:
#Розрахунок розміру ефекту на основі очікуваних показників
effect_size = sms.proportion_effectsize(0.20, 0.19)
print(effect_size)

0.025241594409087353


In [3]:
#Розрахунок необхідного розміру вибірки
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )

#Округлення до наступного цілого числа
required_n = ceil(required_n)
print(f'Потрібно {required_n} користувачів для проведення А/В тесту')

Потрібно 24638 користувачів для проведення А/В тесту


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [4]:
#Зчитування даних з файлу у змінну
df = pd.read_csv('cookie_cats.csv')

print(df.info())
print('\n')
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB
None


   userid  version  sum_gamerounds  retention_1  retention_7
0     116  gate_30               3        False        False
1     337  gate_30              38         True        False
2     377  gate_40             165         True        False
3     483  gate_40               1        False        False
4     488  gate_40             179         True         True


In [5]:
pd.crosstab(df['version'], df['retention_7'])

retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


In [6]:
#Перевірка, що немає користувачів, які були обрані кілька разів
session_counts = df['userid'].value_counts(ascending=False)
multi_users = session_counts[session_counts > 1].count()

print(f'Є {multi_users} користувачів, які зустрічаються кілька разів у наборі даних.')

Є 0 користувачів, які зустрічаються кілька разів у наборі даних.


In [7]:
#Формування 2 вибірок: контрольної групи та тестової групи
control_group = df[df['version'] == 'gate_30'].sample(n=required_n, random_state=22)
treatment_group = df[df['version'] == 'gate_40'].sample(n=required_n, random_state=22)

ab_test = pd.concat([control_group, treatment_group], axis=0)
ab_test.reset_index(drop=True, inplace=True)

print(ab_test.info())
print('\n')
print(ab_test.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49276 entries, 0 to 49275
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          49276 non-null  int64 
 1   version         49276 non-null  object
 2   sum_gamerounds  49276 non-null  int64 
 3   retention_1     49276 non-null  bool  
 4   retention_7     49276 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 1.2+ MB
None


    userid  version  sum_gamerounds  retention_1  retention_7
0  7540471  gate_30              45         True        False
1  3589138  gate_30              21         True        False
2  3177668  gate_30              14         True        False
3  2133884  gate_30              26        False        False
4   492763  gate_30              39         True         True


In [8]:
ab_test['version'].value_counts()

,count
version,
gate_30,24638
gate_40,24638


In [9]:
#Обчислення середнього, стандартного відхилення та стандартної помилки
retention_rates = ab_test.groupby('version')['retention_7']
retention_rates = retention_rates.agg(['mean', 'std', stats.sem])
retention_rates.columns = ['retention_rate', 'std_deviation', 'std_error']


retention_rates.style.format('{:.5f}')

,retention_rate,std_deviation,std_error
version,,,
gate_30,0.18914,0.39163,0.00249
gate_40,0.17798,0.38250,0.00244


Коефіцієнт утримання у версії "ворота на 30 рівні" = 18,91%

Коефіцієнт утримання у версії "ворота на 40 рівні" = 17,8%

Отже, можемо сформулювати гіпотезу, що краще утримання через 7 днів після встановлення гри дає версія "ворота на 30 рівні" (gate_30).


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [10]:
#Тестування гіпотези (z-тест)
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control_results = ab_test[ab_test['version'] == 'gate_30']['retention_7']
treatment_results = ab_test[ab_test['version'] == 'gate_40']['retention_7']

n_con = control_results.count()
n_treat = treatment_results.count()
successes = [control_results.sum(), treatment_results.sum()]
nobs = [n_con, n_treat]

#print(successes)

z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z-статистика: {z_stat:.2f}')
print(f'p-value: {pval:.4f}')
print(f'Довірчий інтервал 95% для групи control (gate_30): [{lower_con:.4f}, {upper_con:.4f}]')
print(f'Довірчий інтервал 95% для групи treatment (gate_40): [{lower_treat:.4f}, {upper_treat:.4f}]')


z-статистика: 3.20
p-value: 0.0014
Довірчий інтервал 95% для групи control (gate_30): [0.1842, 0.1940]
Довірчий інтервал 95% для групи treatment (gate_40): [0.1732, 0.1828]


**Висновок:**

1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?

Оскільки $p$-значення = 0.0014 набагато нижче за $\alpha$ = 0.05, маємо статистично значущу різницю. Отже, різниця між групами не є випадковою.

2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?

Довірчі інтервали:

control group (gate_30):  *[0.1842, 0.1940]*

treatment group (gate_40):  *[0.1732, 0.1828]*

Довірчі інтервали утримання користувачів не перетинаються, є чіткий розрив між результатами. Інтервал тестової групи закінчується на 0,1828, а інтервал контрольної групи починається на 0,1842. Отже, версія з воротами на 30 рівні працює краще, ніж версія з воротами на 40 рівні, і дає краще утримання користувачів через 7 днів після встановлення гри.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


**Гіпотези:**

* $H_0$: утримання гравця на 7 день після реєстрації однакове в обох групах (gate_30 та gate_40).
* $H_a$: є різниця між групами (gate_30 та gate_40) в утриманні гравця на 7 день після реєстрації.

In [11]:
ab_test.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,7540471,gate_30,45,True,False
1,3589138,gate_30,21,True,False
2,3177668,gate_30,14,True,False
3,2133884,gate_30,26,False,False
4,492763,gate_30,39,True,True


In [12]:
#Таблиця спряженості
pd.crosstab(ab_test['version'], ab_test['retention_7'])

retention_7,False,True
version,,
gate_30,19978,4660
gate_40,20253,4385


In [13]:
#Таблиця спряженості
crosstab_chi = pd.crosstab(ab_test['version'], ab_test['retention_7'])

#Тест Хі-квадрат
chi2, p, dof, expected = stats.chi2_contingency(crosstab_chi)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 10.166
p-value = 0.00143
Ступені свободи = 1
Очікувані частоти:
 [[20115.5  4522.5]
 [20115.5  4522.5]]


**Висновок:**

Оскільки $p = 0.00143 < 0.05$, відхиляємо $H_0$.
Отже, є статистично значущою різниця в утриманні між групами gate_30 та  gate_40.